In [1]:
import os
import shutil
from google.colab import drive
drive.mount('/content/drive')

# Define model name and paths
model_name = "gemma4-e4b" # change this
drive_folder = "/content/drive/MyDrive/Models/finetunes/" + model_name + "/"
drive_gguf_path = drive_folder + "gemma-4-e4b-it.Q8_0" + ".gguf" # and this
dataset = "dass_valid_dataset.json" # and this
drive_dataset_path = "/content/drive/MyDrive/Models/" + dataset

local_gguf_path = "/content/" + model_name + ".gguf"
local_dataset_path = "/content/" + dataset

# Copy GGUF file to local runtime
print(f"Copying model from Google Drive to local runtime...")
if not os.path.exists(local_gguf_path):
    shutil.copy2(drive_gguf_path, local_gguf_path)

# Copy dataset to local runtime
print(f"Copying dataset from Google Drive to local runtime...")
if os.path.exists(drive_dataset_path):
    shutil.copy2(drive_dataset_path, local_dataset_path)
else:
    print("Warning: Dataset not found in Drive directory.")

print("Copy complete!")

Mounted at /content/drive
Copying model from Google Drive to local runtime...
Copying dataset from Google Drive to local runtime...
Copy complete!


In [2]:
!sudo apt update
!sudo apt install -y pciutils zstd
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,644 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,292 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,602 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,341 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:13 http://security.ubuntu.com/ubu

In [3]:
%pip install openai scikit-learn pandas

In [4]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

In [5]:
# Create the Modelfile pointing to the local file
modelfile_content = f"FROM {local_gguf_path}\n"
with open("Modelfile", "w") as f:
    f.write(modelfile_content)

# Create the model in Ollama
!ollama create {model_name} -f /content/Modelfile

# List available models to verify
!ollama list


NAME                 ID              SIZE      MODIFIED               
gemma4-e4b:latest    0422e4b501ec    8.0 GB    Less than a second ago    


In [6]:
# Create client

from openai import OpenAI
from sklearn.model_selection import train_test_split
import pandas as pd

client = OpenAI(
    base_url="http://127.0.0.1:11434/v1",  # Ollama
    api_key="sk-1234"  # Dummy key
)

In [7]:
# Set model parameters

SYSTEM_PROMPT = """
You are a DASS-21 survey scoring assistant.
Your job is to extract answers for the DASS-21 based on a given conversation transcript.
The Depression Anxiety Stress Scales 21-item version (DASS-21) is a clinical assessment tool that measures the severity of depression, anxiety, and stress over the past week.

The conversation asks questions in this specific order. Map each user response to the correct q number:
q1  (Stress):      Hard to wind down or switch off
q6  (Stress):      Over-reacting to situations
q8  (Stress):      Using a lot of nervous energy
q11 (Stress):      Getting agitated
q12 (Stress):      Difficult to relax
q14 (Stress):      Impatient or intolerant
q18 (Stress):      Touchy or irritable
q2  (Anxiety):     Dryness of mouth
q4  (Anxiety):     Difficulty breathing
q7  (Anxiety):     Trembling
q9  (Anxiety):     Worried about panicking or embarrassing self
q15 (Anxiety):     Close to panic
q19 (Anxiety):     Aware of heart beating without physical exertion
q20 (Anxiety):     Scared without a clear reason
q3  (Depression):  Couldn't experience any positive feeling
q5  (Depression):  Difficult to work up initiative
q10 (Depression):  Nothing to look forward to
q13 (Depression):  Down-hearted or blue
q16 (Depression):  Unable to become enthusiastic
q17 (Depression):  Not worth much as a person
q21 (Depression):  Life felt meaningless

Scale mapping:
0 = never
1 = sometimes
2 = often
3 = almost always

Input Format:
You will take input in the form of a transcript in JSON matching OpenAI's API format.
Example:
[
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    // ...
]

Output Format:
The final normalized answer for each question must be a single string value: "0", "1", "2", or "3".
Rules:
- Read each user response in order. The first user response answers q1, the second answers q6, the third answers q8, and so on following the order above.
- Score each response based solely on what the user said.
- Return ONLY valid JSON.
- Do not include markdown fences.
- Do not include explanation.
- Do not omit any q1-q21 key.
Return JSON in exactly this shape:
{
  "q1": "0",
  "q6": "0",
  "q8": "0",
  "q11": "0",
  "q12": "0",
  "q14": "0",
  "q18": "0",
  "q2": "0",
  "q4": "0",
  "q7": "0",
  "q9": "0",
  "q15": "0",
  "q19": "0",
  "q20": "0",
  "q3": "0",
  "q5": "0",
  "q10": "0",
  "q13": "0",
  "q16": "0",
  "q17": "0",
  "q21": "0"
}
"""
MODEL = model_name

- Use a json dataset of DASS-21 conversation transcripts
- Extract scores using LLM
- Validate whether they were correct using MSE

In [8]:
# Load the DASS-21 dataset
import json

with open('dass_valid_dataset.json') as f:
    data = json.loads('\n'.join([line for line in f]))

# Split the data into conversation data and expected values
conversations = [entry['conversation'] for entry in data]
expected_outputs = [entry['expected_output'] for entry in data]

print('SUCCESS: Number of conversations matches number of scores' if len(conversations) == len(expected_outputs) else 'FAIL: Number of conversations does not match number of scores')
print(f'Total conversations: {len(conversations)}')

SUCCESS: Number of conversations matches number of scores
Total conversations: 10


In [9]:
# Further split the data into training and test data
conv_train, _conv_test, score_train, _score_test = train_test_split(conversations, expected_outputs, random_state=1234)
print(f'Train: {len(conv_train)}, Test: {len(_conv_test)}')

Train: 7, Test: 3


In [10]:
def Message(content, role='user'):
    return {
        'role': role,
        'content': content
    }

def create_prediction(transcript) -> str:
    completion = client.chat.completions.create(
        model=MODEL,
        messages=[
            Message(SYSTEM_PROMPT, 'system'),
            Message(json.dumps(transcript))
        ],
    )
    return completion.choices[0].message.content

In [11]:
# Run predictions on training and test set
def generate_scores(conversations):
    scores = []
    for i, c in enumerate(conversations):
        print(f'Generating {i+1}/{len(conversations)}')
        scores.append(create_prediction(c))
    return scores

train_scores_raw = generate_scores(conv_train)
test_scores_raw = generate_scores(_conv_test)

Generating 1/7
Generating 2/7
Generating 3/7
Generating 4/7
Generating 5/7
Generating 6/7
Generating 7/7
Generating 1/3
Generating 2/3
Generating 3/3


In [12]:
import re

# Parse json scores
def parse_scores(raw_scores):
    parsed = []
    for s in raw_scores:
        cleaned_s = re.sub(r'<think>.*?</think>', '', s, flags=re.DOTALL).strip()
        try:
            parsed.append(json.loads(cleaned_s))
        except json.JSONDecodeError:
            print(f'Failed to parse: {cleaned_s}')
            parsed.append({f'q{i}': '0' for i in range(1, 22)})
    return parsed

train_scores_parsed = parse_scores(train_scores_raw)
test_scores_parsed = parse_scores(test_scores_raw)

In [13]:
from sklearn.metrics import mean_squared_error

def convert_scores_to_array(scores_df):
    return [int(s) if s else 0 for s in scores_df.values.flatten().tolist()]

def evaluate_split(parsed_scores, expected_scores, name):
    pred_df = pd.DataFrame(parsed_scores)
    expected_df = pd.DataFrame(expected_scores)

    cols = [f'q{i}' for i in range(1, 22)]
    pred_df = pred_df[cols]
    expected_df = expected_df[cols]

    mse = mean_squared_error(
        convert_scores_to_array(expected_df),
        convert_scores_to_array(pred_df)
    )
    print(f'{name} MSE: {mse}')

    display(expected_df.compare(pred_df, align_axis=1, keep_shape=True, keep_equal=True).rename(
        columns={'self': 'Expected', 'other': 'Predicted'}, level=1
    ))

evaluate_split(train_scores_parsed, score_train, 'Train')
evaluate_split(test_scores_parsed, _score_test, 'Test')

Train MSE: 0.0


q1                 q2                 q3                 q4            \
  Expected Predicted Expected Predicted Expected Predicted Expected Predicted   
0        3         3        0         0        1         1        0         0   
1        1         1        0         0        0         0        0         0   
2        2         2        0         0        2         2        1         1   
3        3         3        1         1        2         2        2         2   
4        1         1        0         0        1         1        1         1   
5        2         2        1         1        0         0        1         1   
6        2         2        0         0        3         3        1         1   

        q5            ...      q17                q18                q19  \
  Expected Predicted  ... Expected Predicted Expected Predicted Expected   
0        1         1  ...        0         0        1         1        1   
1        1         1  ...        0         0        0         0        0   
2        1         1  ...        1         1        1         1        1   
3        2         2  ...        2         2        1         1        1   
4        1         1  ...        0         0        1         1        1   
5        1         1  ...        0         0        1         1        1   
6        3         3  ...        2         2        0         0        1   

                 q20                q21            
  Predicted Expected Predicted Expected Predicted  
0         1        0         0        0         0  
1         0        0         0        0         0  
2         1        1         1        1         1  
3         1        1         1        2         2  
4         1        0         0        0         0  
5         1        1         1        0         0  
6         1        0         0        2         2  

[7 rows x 42 columns]

Test MSE: 0.0


q1                 q2                 q3                 q4            \
  Expected Predicted Expected Predicted Expected Predicted Expected Predicted   
0        1         1        2         2        0         0        2         2   
1        2         2        2         2        1         1        3         3   
2        2         2        1         1        2         2        1         1   

        q5            ...      q17                q18                q19  \
  Expected Predicted  ... Expected Predicted Expected Predicted Expected   
0        0         0  ...        0         0        1         1        2   
1        1         1  ...        1         1        1         1        2   
2        2         2  ...        2         2        2         2        1   

                 q20                q21            
  Predicted Expected Predicted Expected Predicted  
0         2        0         0        0         0  
1         2        1         1        1         1  
2         1        0         0        1         1  

[3 rows x 42 columns]